<a href="https://colab.research.google.com/github/wetherc/data-2000/blob/sp26/homework/060_neural-networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework Assignment: Feed-Forward Neural Networks for Tabular Data

**Objective:** In this assignment, you will build, train, and evaluate a Feed-Forward Neural Network (Multi-Layer Perceptron) to process tabular data. You will use the **MovieLens 100k** dataset to predict user ratings for movies based on various features.

### Instructions
1. Run the provided starter code to load the dataset.
2. Complete the tasks outlined in the markdown cells below.
3. Ensure your code is well-commented and your plots are clearly labeled.
4. Answer any conceptual questions in a separate markdown block.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf.keras.utils.set_random_seed(42)

In [ ]:
dataset, ds_info = tfds.load(
    'movielens/100k-ratings',
    split='train',
    with_info=True,
)

# Display some dataset metadata
print(f"\nNumber of examples: {ds_info.splits['train'].num_examples}")
print(f"Dataset features: {list(ds_info.features.keys())}")

## Task 1: Data Exploration
Before building a model, it is crucial to understand your tabular data.

**Your Task:**
1. Extract a batch of records from the dataset (e.g., using `dataset.take(5)`).
2. Print the features and their corresponding values to understand the structure of the data.
3. Identify the target variable (`user_rating`) and continuous/categorical features you might want to use.

In [ ]:
records = []
for example in dataset.take(5):
    # Convert each tensor value to a numpy value
    record = {feature_name: feature_value.numpy() for feature_name, feature_value in example.items()}
    records.append(record)

df = pd.DataFrame(records)
df

## Task 2: Data Preprocessing


### Step 1: Extract Specific Features
First, we isolate the specific features we want our neural network to learn from. We extract `user_zip_code`, `user_gender`, `raw_user_age`, and `movie_genres` as our inputs, and `user_rating` as our target variable. We cast numerical and boolean fields to `tf.float32` for compatibility with TensorFlow.

In [ ]:
def preprocess_data(features):
    inputs = {
        'user_zip_code': features['user_zip_code'],
        'user_gender': tf.cast(features['user_gender'], tf.float32),
        'raw_user_age': tf.cast(features['raw_user_age'], tf.float32),
        'movie_genres': features['movie_genres']
    }
    target = features['user_rating']
    return inputs, target

processed_dataset = dataset.map(preprocess_data)

### Step 2: Build Vocabularies and Normalize Data
Neural networks require numerical inputs, so we cannot pass raw strings or unscaled numbers directly.
- **Categorical Data**: We use a `Hashing` layer to convert high-cardinality string categories (like `user_zip_code`) into a fixed number of integer bins (e.g., 1000 bins). This avoids the need to compute and store a large vocabulary.
- **Multi-Categorical Data**: Since movies can have multiple genres, we use `IntegerLookup` with `multi_hot` encoding to represent genres as a multi-hot array. We can significantly speed up the .adapt() call by batching the dataset before passing it to the method. Processing the data in large batches (e.g., batch(10000)) allows TensorFlow to vectorize the operations and is much more efficient than processing one record at a time. Because the number of genres each movie has can vary (we call these ragged tensors), we will use the `.ragged_batch()` method.
- **Continuous Data**: We use a `Normalization` layer to scale continuous variables like `raw_user_age` so they have a mean of 0 and standard deviation of 1, which helps the network learn faster.

In [ ]:
zip_code_hashing = tf.keras.layers.Hashing(num_bins=1000)

In [ ]:
# Movie Genres Lookup (Multi-Hot)
genre_lookup = tf.keras.layers.IntegerLookup(output_mode='multi_hot')
genre_lookup.adapt(dataset.map(lambda x: x['movie_genres']).ragged_batch(10000))

In [ ]:
# Age Normalization
age_normalization = tf.keras.layers.Normalization(axis=None)
age_normalization.adapt(
    dataset.map(lambda x: tf.cast(x['raw_user_age'], tf.float32)).batch(10000)
)

In [ ]:
# Apply the transformations
def encode_features(inputs, target):
    encoded_inputs = {
        'user_zip_code': zip_code_hashing(inputs['user_zip_code']),
        'user_gender': inputs['user_gender'],
        'raw_user_age': age_normalization(inputs['raw_user_age']),
        'movie_genres': genre_lookup(inputs['movie_genres'])
    }
    return encoded_inputs, target

encoded_dataset = processed_dataset.map(encode_features)

### Step 3: Train/Test Split
To properly evaluate our model, we must test it on data it hasn't seen during training. We shuffle the dataset randomly and split it: 80% for training and 20% for testing.

In [ ]:
num_examples = ds_info.splits['train'].num_examples
train_size = int(0.8 * num_examples)

encoded_dataset = encoded_dataset.shuffle(10000, seed=42)

train_dataset = encoded_dataset.take(train_size)
test_dataset = encoded_dataset.skip(train_size)

### Step 4: Batching and Prefetching
Finally, we group our data into batches (e.g., 32 records at a time) for more efficient memory usage during training.

We also use `.cache()` to keep the data in memory after the first epoch, and `.prefetch(tf.data.AUTOTUNE)` to allow the CPU to prepare the next batch of data while the GPU/CPU is simultaneously training the model on the current batch.

In [ ]:
batch_size = 32

train_dataset = train_dataset.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)

print(f"Zip Code Bins: {zip_code_hashing.num_bins}")
print(f"Movie Genres Vocabulary Size: {genre_lookup.vocabulary_size()}")
print("Datasets successfully prepared, batched, and prefetched.")

## Task 3: Build the Feed-Forward Neural Network
Now you will design the architecture of your neural network for regression.

**Your Task:**
1. Use the Keras Functional API to build a feed-forward neural network.
2. Ensure your input layers correctly handle the shapes of the features you extracted. For categorical features that have been converted to integers (like `user_zip_code` which we hashed into bins), you must use an `Embedding` layer to map these integer indices into dense, continuous vectors. Remember to `Flatten` or pool the embedding output to remove the sequence dimension before concatenating it with your continuous and multi-hot features.
3. Add at least two `Dense` hidden layers with ReLU activation functions. Experiment with the number of neurons.
4. Add a final `Dense` output layer. Since this is a regression task predicting a single continuous value (`user_rating`), how many units should the output layer have, and what should the activation function be?

### Example Usage of Embedding and Flatten Layers
Here is a quick example of how you might process a categorical integer input using the Functional API before concatenating it with other features:

```python
# Define the input (shape is (1,) for a single
# categorical value per record)
categorical_input = tf.keras.Input(
    shape=(1,),
    name='example_cat_feature',
    dtype=tf.int64)

# Apply Embedding, in this case using the
# 1000 bins we created with our ZIP Code hash
# and outputting an embedding dimension of 16
embedding = tf.keras.layers.Embedding(
    input_dim=1000,
    output_dim=16)(categorical_input)
# The output shape here is (batch_size, 1, 16)

# Flatten to remove the extra sequence dimension
flattened_embedding = tf.keras.layers.Flatten()(embedding)
# The output shape is now (batch_size, 16), and is
# ready to be concatenated with the other inputs
```

### Model Code

## Task 4: Model Compilation, Training, and Evaluation

**Your Task:**
1. **Compile the model:** Use the Adam optimizer and Mean Squared Error (`mse`) or Mean Absolute Error (`mae`) for the loss function.
2. **Train the model:** Call `model.fit()` on your training dataset for at least 20 epochs. Use the test dataset as validation data.
3. **Plot training curves:** Extract the loss metric from the training history and plot it over the epochs.
4. **Evaluate:** Print out the final Mean Absolute Error (MAE) or Mean Squared Error (MSE) on the test dataset.